# Fama Macbeth Regression

Modernized for Python 3.10+ without legacy optional dependencies.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
FOLDER = Path.cwd().resolve()
if str(FOLDER) not in sys.path:
    sys.path.insert(0, str(FOLDER))
from linear_model_utils import regression_data, classification_data, panel_factor_data, ols_numpy, rank_ic
OUTPUT = FOLDER.parent / 'data' / 'linear_models'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [2]:
panel = panel_factor_data()
factors = [c for c in panel.columns if c.startswith('factor_')]
rows = []
for date, group in panel.groupby('date'):
    coefs = ols_numpy(group[factors], group['return_1d'])
    rows.append({'date': date, **coefs.to_dict()})
fm = pd.DataFrame(rows)
summary = fm.drop(columns='date').agg(['mean','std']).T.assign(t_stat=lambda df: df['mean'] / (df['std'] / np.sqrt(len(fm))))
fm.to_parquet(OUTPUT / 'fama_macbeth_coefficients.parquet', index=False)
summary.to_csv(OUTPUT / 'fama_macbeth_summary.csv')
display(summary.sort_values('t_stat', key=lambda s: s.abs(), ascending=False).head(10))

,mean,std,t_stat
factor_00,0.002659,0.018238,1.844046
factor_03,0.000958,0.019121,0.633945
factor_02,0.000847,0.019527,0.548973
factor_01,0.000600,0.016920,0.448688
intercept,0.000181,0.005909,0.387067
factor_04,0.000184,0.022990,0.101250
